# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkusmade/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**My lane: content refresh prioritization — ranking/scoring.**
Question: *out of 30,000 live pages, which ~50 should an editor refresh first this week?*

Skills loaded for this card: `framing-ml-problems` + `flyrank/flyrank-data` (see `skills/README.md`).
Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Runtime → Run all works in Colab (clones this repo + installs requirements) and locally (finds the repo root).

## 1. My lane as an ML task (type)

**Ranking / scoring.** My question is a *"which ones first?"* question, so the output is an ordered
queue with a priority score per page. Underneath the ranking sits a binary label (is this page
declining?), so the *mechanism* is classification — but nobody acts on a yes/no verdict for all
30,000 pages. The editor acts on the **top of the list**, so the task is ranking and the metric has
to be measured at the top (section 3).

Not clustering (I am not discovering kinds of pages) and not plain signal analysis (I am not asking
"which signals travel together") — I have a decision to improve.

The four questions from the framing skill, answered in my own words:

1. **Decision this improves** — the weekly refresh queue: which pages get one of the ~50 editor slots.
2. **Who acts, and what they do** — a content editor opens the queue top-down and rewrites/updates
   the page ranked highest.
3. **Cost of a wrong call** — a wasted editor slot means hours spent rewriting a page that was fine.
   A missed declining page loses impressions slowly, over weeks. **False positives are the expensive
   error here**, which is why precision at small K beats recall.
4. **Why data/ML at all** — the signals that a page is declining are many, weak, and point in
   different directions (measured in section 5). Too tangled for one if-statement; exactly what a
   model combines.

In [1]:
# --- Setup: works in Colab (clones this repo) and locally (walks up to the repo root) ---
import os, sys, subprocess

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mayurkusmade/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"could not find {DATA_PATH} from {os.getcwd()}"

df = pd.read_csv(DATA_PATH)
RAW_COLUMNS = df.shape[1]  # as shipped; derived columns are added below as we go
print(f"Loaded {DATA_PATH} -> {df.shape[0]:,} rows x {RAW_COLUMNS} columns")
# Section 1 backing numbers: the decision is "who gets a slot", so size the gap.
rows = len(df)
EDITOR_SLOTS_PER_WEEK = 50  # stated assumption; revisited when the baseline is built in ML-07

print(f"corpus pages        : {rows:,} across {df['client_id'].nunique()} clients")
print(f"editor slots / week : {EDITOR_SLOTS_PER_WEEK}  ->  {EDITOR_SLOTS_PER_WEEK / rows:.3%} of the corpus")
print(f"pages per client    : avg {rows / df['client_id'].nunique():,.0f}  "
      f"(median {df.groupby('client_id').size().median():,.0f})")

Loaded data/raw/content_refresh_anonymized.csv -> 30,000 rows x 44 columns
corpus pages        : 30,000 across 32 clients
editor slots / week : 50  ->  0.167% of the corpus
pages per client    : avg 938  (median 567)


## 2. Target or proxy

**Target: `is_declining_label` = 1 when a page's impressions fell by more than 20%** between the
previous 30-day window and the last one — i.e. the shipped `trend_direction == "down"`, which is
computed from `impressions_last_30d` vs `impressions_prev_30d` with a ±20% band
(`new` = no impressions before but some now; `flat` = none in either window).

**Where does that label come from — observed outcome or defined rule? Honestly: both, and the mix
matters.** The two impression counts are *observed*: GSC measured them. The ±20% threshold is
*chosen by someone*. So this is a **defined proxy** — a reasonable operational definition of
"declining", not a natural fact. The framing skill's rule is *"the target must be observed, not
defined"*, so I record the replacement plan now: on the warehouse release (weeks 3+) the label
becomes a genuinely **observed forward window** — features from months t−3…t, label = did impressions
actually drop in month t+1. That removes the arbitrary threshold *and* puts a clean time gap between
features and label.

**Leakage guards written down before any training:**

- `trend_direction` and `trend_pct` are **never features** — the label is derived from them.
- The 90-day totals (`impressions_90d`, `clicks_90d`, …) **overlap the label's 60-day window**; the
  cell below measures how much. That is a real leakage risk, to be handled deliberately in ML-05 /
  ML-09 — not discovered silently after training.
- The ±20% rule is applied portfolio-wide, so a client whose whole inventory fell (or rose) labels
  almost everything the same way. Another reason the train/test split must be **grouped by
  `client_id`** (frozen in ML-08).

In [2]:
# Section 2: rebuild the label from the two OBSERVED windows and check it matches the shipped column.
last, prev = df["impressions_last_30d"], df["impressions_prev_30d"]


def direction_from_windows(r_last, r_prev):
    """The shipped trend_direction rule, reimplemented so the label source is visible."""
    if r_prev == 0 and r_last > 0:
        return "new"
    if r_prev == 0 and r_last == 0:
        return "flat"
    pct = (r_last - r_prev) / r_prev * 100
    if pct > 20:
        return "up"
    if pct < -20:
        return "down"
    return "stable"


rebuilt = pd.Series([direction_from_windows(l, p) for l, p in zip(last, prev)], name="rebuilt")
agreement = (rebuilt == df["trend_direction"]).mean()
print(f"rebuilt trend_direction matches the shipped column on {agreement:.1%} of rows")

df["is_declining_label"] = (rebuilt == "down").astype(int)
print(f"target prevalence: {df['is_declining_label'].mean():.3f} "
      f"({int(df['is_declining_label'].sum()):,} of {len(df):,} pages)")

# How much of the 90-day feature window sits inside the label's 60-day window? (leakage exposure)
overlap_share = ((last + prev) / df["impressions_90d"].replace(0, np.nan)).median()
print(f"median share of impressions_90d that falls inside the label window: {overlap_share:.0%}"
      "  <- handle in ML-05 / ML-09, do not ignore")

rebuilt trend_direction matches the shipped column on 100.0% of rows
target prevalence: 0.542 (16,262 of 30,000 pages)
median share of impressions_90d that falls inside the label window: 57%  <- handle in ML-05 / ML-09, do not ignore


## 3. Success metric

**Primary: Precision@50** — of the 50 pages at the top of the queue, what share actually carries the
declining label. K = 50 because that is the editor capacity from section 1: the metric is measured
where the decision is made, not across the whole corpus.

**The floor: the base rate.** 54.2% of pages in this slice carry the label, so a *random* queue
already scores ≈ 0.54 on average. A score of 0.55 has learned nothing. Every claim has to clear that
floor by a margin that survives a **client-grouped holdout** (split defined and frozen in ML-08 —
chosen before training, not after looking at results).

**What "good" means — defined now, before any training:**

- Beats the hand-rule baseline (`scripts/02_baseline_score.py`, copied verbatim into the cell below)
  at Precision@50 on held-out clients.
- Clears the 0.542 base-rate floor by a clear margin on the same holdout.
- Secondary check: ROC-AUC across all held-out pages, so one lucky top-50 cannot carry the claim.

**Careful words:** this measures how well a queue matches a *rule-defined proxy label* on
*anonymized historical data*. It is decision-support for an editor's queue — not a prediction of
Google's algorithm, and not a causal claim that refreshing a page recovers traffic.

In [3]:
# Section 3: the metric must be computable TODAY, on a baseline. So compute it here.
def precision_at_k(score_column, k=50):
    top = df.sort_values(score_column, ascending=False).head(k)
    return top["is_declining_label"].mean()


base_rate = df["is_declining_label"].mean()
random_p50 = pd.Series(
    [df["is_declining_label"].sample(50, random_state=s).mean() for s in range(500)]
).mean()

print(f"random queue, expected P@50 : {random_p50:.3f}   (equals the base rate {base_rate:.3f})")


# --- Hand-rule baseline, copied verbatim from scripts/02_baseline_score.py (reference pipeline) ---
def _percentile_rank(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)


def _normalize(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = values.min(), values.max()
    return pd.Series(np.zeros(len(values))) if hi == lo else (values - lo) / (hi - lo)


df["visibility_score"] = _percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = _percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - _normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - _percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

print(f"hand-rule baseline, P@50     : {precision_at_k('baseline_refresh_score'):.3f}"
      f"   (weights 0.40/0.30/0.25/0.05 chosen by hand, never measured at K=50)")

print("\nsingle fixed thresholds (a rule must also be SELECTIVE to be usable):")
for column, threshold in [
    ("impressions_90d", 500),
    ("word_count", 1200),
    ("content_age_days", 365),
    ("days_since_last_update", 180),
]:
    flagged = df[column].fillna(0) >= threshold
    print(f"  {column} >= {threshold:<5} precision={df.loc[flagged, 'is_declining_label'].mean():.3f}"
          f"  coverage={flagged.mean():5.1%}  ({int(flagged.sum()):,} pages for an editor to review)")

random queue, expected P@50 : 0.548   (equals the base rate 0.542)
hand-rule baseline, P@50     : 0.340   (weights 0.40/0.30/0.25/0.05 chosen by hand, never measured at K=50)

single fixed thresholds (a rule must also be SELECTIVE to be usable):
  impressions_90d >= 500   precision=0.596  coverage=55.8%  (16,726 pages for an editor to review)
  word_count >= 1200  precision=0.588  coverage=70.0%  (21,007 pages for an editor to review)
  content_age_days >= 365   precision=0.426  coverage=21.2%  (6,360 pages for an editor to review)
  days_since_last_update >= 180   precision=0.471  coverage= 0.6%  (174 pages for an editor to review)


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page) for one client, summarized over one trailing 90-day
window** — 30,000 rows × 44 columns, 32 pseudonymized clients, `content_id` unique per row.
Not one row per day (that lives in the ~79M-row warehouse release, weeks 3+), and not one row per
query.

That unit choice decides what I can claim: with one snapshot per page I can rank pages **today**;
I cannot yet say "this page started declining last Tuesday". The forward-window label of section 2
gets built on the daily table.

Gotchas from `flyrank-data` that are visible in this slice (all printed below):
`avg_position = 0` is a *no position data* sentinel (1,205 rows); `feedly article` rows carry **no
keyword data at all** (100% missing `search_volume`); `word_count` is blank on 7,699 rows.
Missingness follows `content_type`, so a blind `fillna(0)` would smuggle content type into the
features — the data contract in ML-04 adds `has_`-flags instead.

In [4]:
# Section 4: the lane's slice, shown as the actual dataframe it is.
print(f"as shipped: {df.shape[0]:,} rows x {RAW_COLUMNS} columns | working frame now has "
      f"{df.shape[1]} columns (label + baseline scores added by the cells above)")
print(f"clients: {df['client_id'].nunique()} | unique content_id: {df['content_id'].nunique():,} | "
      f"duplicate rows on content_id: {int(df['content_id'].duplicated().sum())}")

# Missingness follows content_type — this is why a blind fillna(0) is dangerous.
missing_by_type = df.groupby("content_type")[
    ["search_volume", "competition", "cpc", "word_count", "main_intent"]
].apply(lambda block: block.isna().mean().round(3))
print("\nmissing rate by content_type:")
display(missing_by_type)

print(f"\navg_position == 0 sentinel ('no position data', not rank zero): "
      f"{int((df['avg_position'] == 0).sum()):,} rows")

SHOW = [
    "content_id", "client_id", "content_type", "main_intent", "word_count",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "engagement_rate",
    "days_since_last_update", "impressions_last_30d", "impressions_prev_30d",
    "trend_direction", "is_declining_label",
]
print("\nfirst 5 rows of the lane slice (IDs are pseudonyms — grouping only, never features):")
display(df[SHOW].head(5))

print("one whole record, transposed — this is the row an editor's queue points at:")
display(df.loc[df.index[0], SHOW].to_frame(name="value"))

as shipped: 30,000 rows x 44 columns | working frame now has 50 columns (label + baseline scores added by the cells above)
clients: 32 | unique content_id: 30,000 | duplicate rows on content_id: 0

missing rate by content_type:


,search_volume,competition,cpc,word_count,main_intent
content_type,,,,,
comparison article,0.000,0.000,0.000,0.000,0.00
feedly article,1.000,1.000,1.000,0.000,1.00
keyword article,0.014,0.014,0.014,0.283,0.01



avg_position == 0 sentinel ('no position data', not rank zero): 1,205 rows

first 5 rows of the lane slice (IDs are pseudonyms — grouping only, never features):


,content_id,client_id,content_type,main_intent,word_count,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,impressions_last_30d,impressions_prev_30d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,3803,29,0.76,10.6,5.88,20,578,987,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,15320,7,0.05,20.3,0.00,25,2501,5915,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,12581,11,0.09,36.5,0.00,20,2382,6089,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,11751,58,0.49,6.2,1.28,22,3626,4206,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,19140,24,0.13,44.0,0.00,14,4211,6452,down,1


one whole record, transposed — this is the row an editor's queue points at:


,value
content_id,content_304f48230142
client_id,client_f369cb89fc
content_type,keyword article
main_intent,transactional
word_count,3221.0
impressions_90d,3803
clicks_90d,29
ctr,0.76
avg_position,10.6
engagement_rate,5.88


## 5. Why ML beats a fixed rule here

Measured below, not asserted:

1. **No single signal separates the classes.** Univariate ROC-AUC across 13 candidate signals runs
   0.41–0.58 — every one is barely better (or worse) than a coin flip alone, and they point in
   *different* directions (`content_age_days` and `search_volume` are inverted in this slice:
   younger and higher-volume pages decline more often).
2. **The rules that do beat the base rate do it by flagging half the corpus.** `impressions_90d >=
   500` reaches precision ≈ 0.60 — above the 0.542 floor — but fires on 55.8% of pages. An editor
   with 50 slots cannot act on ~16,700 flagged pages. A rule that is "right" but not *selective*
   does not solve the decision.
3. **The hand-tuned weighted rule lands below the floor at K=50.** The reference baseline (fixed
   weights 0.40/0.30/0.25/0.05, chosen by feel) scores ≈ 0.34 Precision@50 on this slice — worse
   than a random queue. That is what hand-tuned thresholds do: they look reasonable and were never
   measured at the top of the queue.
4. **The pattern varies by client.** Declining rates run from 0.00 to 0.94 across the 32 clients,
   across 42 content-type × impression-tier × position-tier strata. A global if-ladder cannot
   express that; a model evaluated with client-grouped splits can at least be *measured* against it.

**Honest claim:** the pattern is real but messy — many weak, differently-signed, interacting
signals. ML earns its place as a *combination of weak signals with a measured, client-held-out
evaluation*. Claim level stays **decision-support**; the actual lift over the baseline gets
measured in ML-08, not assumed here.

In [5]:
# Section 5: the "too messy for an if-statement" claim, measured.
from sklearn.metrics import roc_auc_score

y = df["is_declining_label"]

print("univariate ROC-AUC vs the label (weak + mixed directions = many weak signals):")
auc_rows = []
for column in [
    "impressions_90d", "days_with_impressions", "word_count", "clicks_90d", "sessions_90d",
    "scroll_rate", "ctr", "days_since_last_update", "avg_position", "engagement_rate",
    "ai_traffic_pct", "search_volume", "content_age_days",
]:
    values = pd.to_numeric(df[column], errors="coerce")
    mask = values.notna()
    if column == "avg_position":
        mask &= values > 0  # 0 is the "no position data" sentinel
    if mask.sum() < 100:
        continue
    auc_rows.append((column, roc_auc_score(y[mask], values[mask])))

for column, auc in sorted(auc_rows, key=lambda row: row[1]):
    tag = "  (inverted)" if auc < 0.5 else ""
    print(f"  {column:<24} {auc:.3f}{tag}")
print(f"  range: {min(a for _, a in auc_rows):.3f} - {max(a for _, a in auc_rows):.3f}"
      "  <- no usable single-threshold signal")

print(f"\nsingle fixed thresholds vs the {base_rate:.3f} base-rate floor:")
for column, threshold in [
    ("impressions_90d", 500),
    ("word_count", 1200),
    ("content_age_days", 365),
    ("days_since_last_update", 180),
]:
    flagged = df[column].fillna(0) >= threshold
    print(f"  {column} >= {threshold:<5} precision={df.loc[flagged, 'is_declining_label'].mean():.3f}"
          f"  coverage={flagged.mean():5.1%}")

by_client = df.groupby("client_id")["is_declining_label"].agg(["mean", "count"])
print(f"\nclient-level declining rate: {by_client['mean'].min():.2f} -> {by_client['mean'].max():.2f}"
      f" across {len(by_client)} clients (smallest client: {int(by_client['count'].min())} pages)")
print("strata (content_type x impression_tier x position_tier): "
      f"{df.groupby(['content_type', 'impression_tier', 'position_tier']).ngroups}")

univariate ROC-AUC vs the label (weak + mixed directions = many weak signals):


  content_age_days         0.409  (inverted)
  search_volume            0.439  (inverted)
  avg_position             0.485  (inverted)
  engagement_rate          0.496  (inverted)
  ai_traffic_pct           0.502
  ctr                      0.515
  sessions_90d             0.519
  scroll_rate              0.520
  clicks_90d               0.525
  days_since_last_update   0.527
  word_count               0.546
  days_with_impressions    0.579
  impressions_90d          0.584
  range: 0.409 - 0.584  <- no usable single-threshold signal

single fixed thresholds vs the 0.542 base-rate floor:
  impressions_90d >= 500   precision=0.596  coverage=55.8%
  word_count >= 1200  precision=0.588  coverage=70.0%
  content_age_days >= 365   precision=0.426  coverage=21.2%
  days_since_last_update >= 180   precision=0.471  coverage= 0.6%

client-level declining rate: 0.00 -> 0.94 across 32 clients (smallest client: 3 pages)
strata (content_type x impression_tier x position_tier): 42


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) — executed locally with
      `nbclient`, outputs saved in this file
- [x] No client names, URLs, or private queries anywhere — IDs are pseudonyms, used only for
      grouping and the client-level summary
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Next (ML-04):** turn section 2 and 4 into a written data contract — `has_`-flags instead of
blind `fillna(0)`, feature/label window alignment, and the frozen client-grouped split.